# 2. Calidad de datos

Antes de modelar se revisan los nulos, los duplicados y los valores imposibles o extremos de las 10 predictoras.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import *

set_style()
np.random.seed(SEED)
pd.set_option("display.max_columns", 40)
df = load_data()

## 2.1 Valores nulos

In [ ]:
nulos = df[NUMERIC + [TARGET]].isna().sum().rename("nulos").to_frame()
nulos["porcentaje"] = (100 * nulos["nulos"] / len(df)).round(2)
nulos

Las 10 predictoras y la variable objetivo están **completas**: no hay nulos que imputar. La única columna del dataset con nulos es `area` (27.7 %), que no se usa como predictora.

## 2.2 Duplicados

In [ ]:
resumen_dup = pd.Series({
    "Filas duplicadas exactas": df.duplicated().sum(),
    "Códigos (code) repetidos": df["code"].duplicated().sum(),
    "Filas con coordenadas repetidas": df.duplicated(["latitude", "longitude"]).sum(),
    "Filas con predictoras + objetivo idénticos": df.duplicated(NUMERIC + [TARGET]).sum(),
}, name="n")
resumen_dup.to_frame()

In [ ]:
coords = df[df.duplicated(["latitude", "longitude"], keep=False)]
grupos = coords.groupby(["latitude", "longitude"])
print(f"Ubicaciones repetidas: {grupos.ngroups:,} (abarcan {len(coords):,} filas)")
print(f"Ubicaciones repetidas con clases distintas: {(grupos[TARGET].nunique() > 1).sum()}")

No hay filas idénticas ni códigos repetidos: cada registro es una planta distinta. Sin embargo, **6 830 filas repiten las coordenadas de otra planta** (3 064 ubicaciones compartidas, típicamente parques construidos en varias fases). Como las 10 predictoras dependen de la ubicación, **6 406 filas son idénticas a otra en las predictoras y en la clase**.

Con una partición aleatoria, una copia puede caer en entrenamiento y su gemela en prueba, lo que infla las métricas de prueba. Para la siguiente entrega se recomienda **eliminar esos duplicados antes de la partición** o agrupar la validación por coordenada.

## 2.3 Valores extremos e imposibles

In [ ]:
pd.Series({
    "ghi = 0": (df["ghi"] == 0).sum(),
    "humidity = 0": (df["humidity"] == 0).sum(),
    "wind_speed = 0": (df["wind_speed"] == 0).sum(),
    "optimal_tilt <= 0": (df["optimal_tilt"] <= 0).sum(),
    "elevation < 0": (df["elevation"] < 0).sum(),
}, name="registros").to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.5))
sns.boxplot(x=df["dist_to_road"], ax=ax, color="#9ecae1", fliersize=2)
ax.set_xscale("log")
ax.set_title("dist_to_road (escala log)")
plt.tight_layout(); plt.show()

- **Ceros imposibles:** muy pocos casos (2 o 3 en `ghi`, `humidity` y `wind_speed`). Se tratan como nulos.
- **`optimal_tilt` ≤ 0 (9 plantas):** están casi sobre el ecuador (Kenia, Ecuador, Gabón…), donde la inclinación óptima es cercana a 0°. Son valores reales y se conservan.
- **Elevación negativa (255 plantas):** zonas reales bajo el nivel del mar. Se conserva.
- **`dist_to_road`:** la mayoría de valores son pequeños, pero hay una cola larga hacia la derecha (hasta 2 300 km). No se elimina: se aplica **logaritmo** y se **recorta en el percentil 99**.